# 公式 o11 — MinDE System with Mesoscopic Simulator（空間・meso）

> **出典（E-Cell4 公式）**: Examples / example08 — https://ecell4.e-cell.org/examples/example08.html
>
> 大腸菌の **Min タンパク質の極間振動**（細胞分裂面を中央に決める空間パターン）を、
> **メソスコピック（区画反応拡散）** シミュレータで再現する空間例。
>
> ⚠️ **本ノートは自動実行していません（出力なし）**。理由:
> (1) 空間反応拡散で**重い**（本環境で duration の実時間が数分〜）。
> (2) 可視化 `plotting.plot_world` / `plot_movie` は**インタラクティブ**（ヘッドレスでは描画されない）。
> 手元で `uv run jupyter lab` から実行してください。コードは公式のまま忠実に掲載しています。

In [ ]:
# 公式コード（そのまま）。※重い・可視化はインタラクティブ
from ecell4 import *
from ecell4_base.core import *
from ecell4_base import meso

with species_attributes():
    D | DE | {'D': 0.01, 'location': 'M'}          # 膜(M)上
    D_ADP | D_ATP | E | {'D': 2.5, 'location': 'C'}  # 細胞質(C)

with reaction_rules():
    D_ATP + M > D | 0.0125                          # MinD-ATP が膜へ
    D_ATP + D > D + D | 9e+6 * (1e+15 / N_A)         # 協同的な膜集積
    D + E > DE | 5.58e+7 * (1e+15 / N_A)             # MinE 結合
    DE > D_ADP + E | 0.7                             # 加水分解して脱離
    D_ADP > D_ATP | 0.5                              # 再充填

m = get_model()

w = meso.World(Real3(4.6, 1.1, 1.1), 0.05)          # 桿菌サイズの区画世界
w.bind_to(m)
rod = Rod(3.5, 0.55, w.edge_lengths() * 0.5)         # 桿状の細胞
w.add_structure(Species('C'), rod)                  # 細胞質
w.add_structure(Species('M'), rod.surface())        # 膜
w.add_molecules(Species('D_ATP'), 2001)
w.add_molecules(Species('D_ADP'), 2001)
w.add_molecules(Species('E'), 1040)

sim = meso.Simulator(w)
obs1 = FixedIntervalNumberObserver(0.1, [sp.serial() for sp in m.list_species()])
obs2 = FixedIntervalHDF5Observer(1.0, 'minde%03d.h5')
duration = 120
sim.run(duration, (obs1, obs2))

# 可視化（インタラクティブ）
plotting.plot_world(w, species_list=('D', 'DE'))    # 空間スナップショット
show(obs1)                                          # 時系列
plotting.plot_movie(obs2, species_list=('D', 'DE')) # 動画

## モデルの説明

MinD-ATP が膜に集積し（協同的）、MinE がそれを剥がして加水分解する。細胞質での再充填を挟むことで、
**MinD/MinE の膜局在が細胞の一端 → 反対端 → …と往復振動**する。この極間振動が細胞中央に「くぼみ」を作らず
分裂面を中央に定める。細菌の自己組織化パターン形成の代表例。

**要点（公式が教えたい機能）**: `meso.World`＋`add_structure(Rod / surface)` で**区画化した空間**を作り、
`location` 付き種（膜 M / 細胞質 C）で**反応拡散**を回す。`plot_world`/`plot_movie` で空間・動画を可視化。

> 実行を軽くするには `duration` を小さく（例 30）し、`obs1.data()` の時系列だけを matplotlib で描くとよい。